In [ ]:
pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
from ultralytics import YOLO
import os

# 1. Mount Drive (This connects your Google Drive to Colab)
drive.mount('/content/drive')

# # 2. Define your "Safe House" paths
# # We will save ALL training results directly to Drive, not the temporary Colab storage.
# ROOT_FOLDER = '/content/drive/MyDrive/smart_aquaculture/yolov8_project/predator_detection'
# DATA_YAML = f'{ROOT_FOLDER}/dataset/data.yaml'  # Make sure this points to your yaml
# EXISTING_MODEL = f'{ROOT_FOLDER}/cormorant_fix/model_for_cormorant_fix/first_best.pt' # Start from your current best

# # This is where checkpoints will be saved live
# TRAIN_SAVE_DIR = f'{ROOT_FOLDER}/training_checkpoints'

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Mounted at /content/drive


In [ ]:
# ==============================================================================
# 2. CONFIGURATION PATHS
# ==============================================================================

# Your Main Project Folder
ROOT_FOLDER = '/content/drive/MyDrive/smart_aquaculture/yolov8_project/predator_detection'

# Path to your Data (dataset info)
DATA_YAML = f'{ROOT_FOLDER}/dataset/data.yaml'

# Path to your STARTING Model (Your current best)
START_MODEL = f'{ROOT_FOLDER}/cormorant_fix/model_for_cormorant_fix/first_best.pt'

# Path where we will save the NEW training checkpoints
# We use a dedicated folder so we can easily find the 'last.pt' file later
TRAIN_SAVE_DIR = f'{ROOT_FOLDER}/training_checkpoints'

# Name of this specific training run
PROJECT_NAME = 'retrain_v1_fixed'

# The specific file we look for to resume training
LAST_CHECKPOINT = f'{TRAIN_SAVE_DIR}/{PROJECT_NAME}/weights/last.pt'

print(f"📂 Project Configured!")
print(f"   Start Model: {START_MODEL}")
print(f"   Save Location: {TRAIN_SAVE_DIR}/{PROJECT_NAME}")



📂 Project Configured!
   Start Model: /content/drive/MyDrive/smart_aquaculture/yolov8_project/predator_detection/cormorant_fix/model_for_cormorant_fix/first_best.pt
   Save Location: /content/drive/MyDrive/smart_aquaculture/yolov8_project/predator_detection/training_checkpoints/retrain_v1_fixed


In [ ]:
# ==============================================================================
# 3. SMART TRAINING FUNCTION
# ==============================================================================
def start_smart_training():
    # CHECK: Does an interrupted training session exist?
    if os.path.exists(LAST_CHECKPOINT):
        print(f"🔄 FOUND CHECKPOINT! Resuming training from: {LAST_CHECKPOINT}")
        print("   (Resuming restores the state exactly as it was left)")

        try:
            # Resume existing training
            model = YOLO(LAST_CHECKPOINT)
            model.train(resume=True)
        except Exception as e:
            print(f"❌ Error resuming: {e}")
            print("   If the file is corrupted, delete 'last.pt' from Drive and try again.")

    else:
        print(f"🚀 NO CHECKPOINT FOUND. Starting FRESH training from: {START_MODEL}")

        # Validate that your start model actually exists
        if not os.path.exists(START_MODEL):
            raise FileNotFoundError(f"❌ Cannot find your start model at: {START_MODEL}")

        # Load your starting model
        model = YOLO(START_MODEL)

        # Start Training with "Bad Camera" Simulation
        # These settings mimic the noise/blur of a Raspberry Pi Camera V1
        model.train(
            data=DATA_YAML,
            project=TRAIN_SAVE_DIR,  # Save inside Drive
            name=PROJECT_NAME,       # Folder name

            # --- Hardware Settings ---
            epochs=50,
            imgsz=640,               # Pixelated input (matches Pi)
            batch=16,                # If you crash, change this to 8
            save=True,
            save_period=5,           # Save backup every 5 epochs
            patience=0,              # Force it to finish all 50 epochs

            # --- "BAD CAMERA" AUGMENTATIONS ---
            lr0=0.005,               # Low learning rate (fine-tuning)

            # Lighting Fixes (For dark birds)
            hsv_h=0.015,
            hsv_s=0.7,
            hsv_v=0.6,

            # Angle Fixes (For birds not perfectly straight)
            degrees=15.0,
            translate=0.2,
            scale=0.6,               # Zoom out to see small snakes

            # Quality Fixes (For blur and obstructions)
            flipud=0.5,              # Upside down (reflections)
            fliplr=0.5,              # Left-Right flip
            mosaic=1.0,              # Mixes images together
            erasing=0.4              # Randomly blocks parts of the bird
        )

In [ ]:
TRAINED_NEW_MODEL = f'{TRAIN_SAVE_DIR}/{PROJECT_NAME}/weights/best.pt'
print(TRAINED_NEW_MODEL)

In [ ]:
# ==============================================================================
# 4. EXECUTE
# ==============================================================================
if __name__ == '__main__':
    start_smart_training()

🚀 NO CHECKPOINT FOUND. Starting FRESH training from: /content/drive/MyDrive/smart_aquaculture/yolov8_project/predator_detection/cormorant_fix/model_for_cormorant_fix/first_best.pt
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/smart_aquaculture/yolov8_project/predator_detection/dataset/data.yaml, degrees=15.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.6, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005

In [ ]:
import os
from ultralytics import YOLO
from google.colab import drive

# 1. Setup
drive.mount('/content/drive')

# --- CONFIGURATION ---
# Define the path to your dataset YAML (Must match what you used for training)
ROOT_FOLDER = '/content/drive/MyDrive/smart_aquaculture/yolov8_project/predator_detection'
DATA_YAML = f'{ROOT_FOLDER}/dataset/data.yaml'

# Define the path to your NEW model
TRAIN_SAVE_DIR = f'{ROOT_FOLDER}/training_checkpoints'
PROJECT_NAME = 'retrain_v1_fixed2'
TRAINED_NEW_MODEL = f'{TRAIN_SAVE_DIR}/{PROJECT_NAME}/weights/best.pt'

# 2. Run Validation
if not os.path.exists(TRAINED_NEW_MODEL):
    print(f"❌ Error: Model not found at {TRAINED_NEW_MODEL}")
else:
    print(f"✅ Loading Model: {TRAINED_NEW_MODEL}")
    model = YOLO(TRAINED_NEW_MODEL)

    print("\n📊 Running Validation on Test Data...")
    # This runs the model on your 'val' images and compares answers
    metrics = model.val(data=DATA_YAML, split='val')

    # 3. Print Clean Results
    print("\n" + "="*40)
    print("       🚀 MODEL ACCURACY REPORT       ")
    print("="*40)
    print(f"Precision (Accuracy): {metrics.results_dict['metrics/precision(B)']:.3f}")
    print(f"Recall (Detection):   {metrics.results_dict['metrics/recall(B)']:.3f}")
    print(f"mAP50 (Overall):      {metrics.results_dict['metrics/mAP50(B)']:.3f}")
    print("="*40)
    print("Interpreting your scores:")
    print(" - Precision: How often 'Snake' is actually a Snake.")
    print(" - Recall:    How many Snakes it found vs. missed.")
    print(" - mAP50:     The general grade (0.90+ is an 'A').")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Loading Model: /content/drive/MyDrive/smart_aquaculture/yolov8_project/predator_detection/training_checkpoints/retrain_v1_fixed2/weights/best.pt

📊 Running Validation on Test Data...
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.9±0.4 ms, read: 0.1±0.0 MB/s, size: 43.0 KB)
val: Scanning /content/drive/MyDrive/smart_aquaculture/yolov8_project/predator_detection/dataset/valid/labels.cache... 242 images, 42 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 242/242 46.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 3.0it/s 5.3s
                   all        242        271      0.834      0.813      0.862      0.565
             cormorant        